# T0.3 — Session-wise SLAP2 / HARP / BonVision synchronization

**Question for Kaspar:** *Across the recorded sessions, how consistently can we identify stimulus transitions in HARP, relate these to BonVision timestamps, and establish when SLAP2 acquired physiology relative to those transitions? Are measured errors below one display frame?*

This is a **read-only, session-registry-driven** audit. Every figure saves as a 300-dpi PNG, SVG, and accompanying `.txt` explaining **source, statistic, interpretation, and limitation**. No voltage traces are smoothed or filtered.

**Timing links and their evidential status:**

| Link | What is measured? | What does it establish? |
|---|---|---|
| **BonVision → HARP photodiode** | Timestamp of BonVision predicted transitions (after the already-saved correction) minus threshold-detected HARP analog transitions; per-event residuals, timing drift, and within-frame fraction. Modern encoded-PD sessions also have held-out predictions. | Timing *fit consistency*; modern held-out points give stronger predictive evidence. Not independent absolute source-sample mapping. |
| **HARP photodiode ↔ HARP DI3** | Signed time of each observed stimulus transition relative to its nearest DI3 acquisition-clock rising edge, using original HARP indices. | Physical phase of actual stimulus transitions in the HARP acquisition clock. **Not** a timing error; phase is bounded by half a cycle by construction. Valid only if analog and digital index origins are genuinely shared. |
| **SLAP2 line indices → HARP DI3** | Requires verified source line-index/cycle association, line period, epoch and DMD timing anchor. | Necessary to certify **sample-level** imaging↔stimulus accuracy. Cannot be inferred from a uniform derived `timebase_sec`. An optional independently verified CSV may supply this evidence. |

**Older sessions:** Legacy `frame_modclass_affine_anchored_edges` measurements are **in-fit consistency** and may have periodic phase ambiguities. Report separately from modern `bv_photodiode_*` measurements. `existing_corrected_timestamps` is *reuse provenance*, not an alignment method: the original method is recovered from `bonsai_event_log.csv` where present.

**Acceptance:** Source sample-to-stimulus p95 absolute timing error below **one verified display frame**. A per-session behavior check compared with an *estimated/reference* frame is **not** a T0.3 PASS. Disposition is PASS, FAIL, or NOT_INDEPENDENTLY_VERIFIED. A missing result is not a failure.

In [ ]:
%matplotlib notebook

In [ ]:
from pathlib import Path
import sys, warnings
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from vip_slap2_analysis.utils.utils import save_figure

# Works inside a normal editable install, or from notebooks/qc/ in a repo checkout.
for parent in [Path.cwd(), *Path.cwd().parents]:
    if (parent/'src'/'vip_slap2_analysis'/'io'/'session_registry.py').is_file():
        sys.path.insert(0,str(parent/'src'))
        break
from vip_slap2_analysis.metadata.t03_session_sync import run_registry_audit
from vip_slap2_analysis.metadata.t03_event_metrics import extend_dataset_audit

plt.rcParams.update({'axes.spines.top':False,'axes.spines.right':False,
                     'font.size':10,'figure.figsize':(10,4.8),'savefig.dpi':300})
print('Python:',sys.version.split()[0], '| Matplotlib:',plt.matplotlib.__version__)
print('Matplotlib backend:',plt.get_backend())
print('If %matplotlib notebook does not work in your JupyterLab frontend, use classic Notebook or %matplotlib widget (ipympl).')

from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

## 0. Configuration and session selection

Set the canonical registry Excel path. `SESSION_IDS=None` audits the entire registry; a one-element list is a fast pilot. `DETAIL=True` is **required** for event-level timing and DI3 measurements. `SCAN_EXTRACTION=True` adds voltage H5 and GIAnT sample-index QC and can require extensive network I/O.

`DISPLAY_HZ_REFERENCE=60` provides only a **visual reference of 16.67 ms** where a session lacks a display-rate estimate. The estimated rate in `alignment_meta.json` is based on the photodiode transition period and an assumed 30 frames per transition. Neither is independently verified. Use a separately reviewed validation file to establish the formal threshold.

In [ ]:
SUMMARY_XLSX = Path(r"\\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\VIP_SD_summary.xlsx")
SESSION_IDS = None  # e.g. ['863774_2026-08-04_08-12-47']
DETAIL = True
SCAN_EXTRACTION = True  # turn on for duplicate sample / GIAnT frame-line-index QC
DISPLAY_HZ_REFERENCE = 60.0
MAX_EVENT_MATCH_MS = 80.0  # one-to-one event association tolerance; never a QC pass threshold
VALIDATED_CLOCK_CSV = None  # Optional independently reviewed SLAP2 line -> HARP DI3 evidence CSV
OUT = Path.cwd()/'T03_sync_audit_outputs_v2'
FOCUS_SESSION = '863774_2026-08-04_08-12-47'

assert SUMMARY_XLSX.is_file(), f'Registry missing: {SUMMARY_XLSX}'
assert DETAIL, 'Turn on DETAIL to measure per-event BonVision/HARP and HARP/DI3 timing.'
OUT.mkdir(parents=True, exist_ok=True)
(OUT/'figures').mkdir(exist_ok=True)
REF_MS=1000/DISPLAY_HZ_REFERENCE
print('Reference frame duration:',round(REF_MS,3),'ms; not independently verified')
print('Output directory:',OUT.resolve())

## 1. Collect the original registry-driven timing QC and direct event residuals

Reads the registry-selected session folders, cached HARP digital pulses (`HARP_df.csv`), analog photodiode (`photodiode.pkl`), corrected Bonsai event logs, and `analysis/qc/behavior/alignment_meta.json`. The extension evaluates individual corrected BonVision transitions against measured HARP PD edges. **No alignment or data extraction is rerun**. It retains sessions without necessary files as missing evidence.

**Sign convention:** BonVision-to-HARP event residual = `corrected_BV_event_time − measured_HARP_PD_time`, in ms. Positive means the corrected BonVision prediction is later than the measured edge. The fitted BonVision/HARP intercept is a *clock-coordinate parameter*, not a stimulus latency.

In [ ]:
df_base, event_detail = run_registry_audit(
    SUMMARY_XLSX,selection=SESSION_IDS,detail=DETAIL,
    include_h5=SCAN_EXTRACTION,validation_csv=VALIDATED_CLOCK_CSV,output_dir=OUT)
df, event_arrays = extend_dataset_audit(
    df_base,event_detail,SUMMARY_XLSX,frame_period_ms=REF_MS,
    max_match_ms=MAX_EVENT_MATCH_MS,output_dir=OUT)
print('Sessions:',len(df), '| method counts:',df['alignment_pathway'].value_counts(dropna=False).to_dict())
print('Behavioral event checks:',df['behavior_event_gate'].value_counts(dropna=False).to_dict())
print('Independent T0.3:',df['t03_status'].value_counts(dropna=False).to_dict())
show=['session_id','modality','alignment_pathway','event_match_n','event_coverage',
      'event_median_ms','event_p95_abs_ms','event_fraction_within_frame',
      'bv_harp_holdout_p95_ms','harp_photodiode_to_di3_p95_abs_phase_ms',
      'di3_period_ms','behavior_event_gate','t03_status','analysis_errors']
display(df[[c for c in show if c in df]].head(15))

## Figure exports and shared display conventions

Each figure uses **session-level** estimates for the across-session distributions, so recordings with more photodiode transitions do not silently dominate. Event-weighted plots are explicitly identified. The dashed one-frame reference uses the estimated per-session frame duration where available or the configured nominal fallback. It is *not* a validated session acceptance threshold.

In [ ]:
FIGURES=[]

def _nums(v): return pd.to_numeric(v,errors='coerce').to_numpy(dtype=float)

def _none(ax,msg):
    ax.text(.5,.5,msg,ha='center',va='center',transform=ax.transAxes,fontsize=11)
    ax.set_xticks([]);ax.set_yticks([])

def savefig(fig,name,title,source,method,meaning,limitation):
    fig.suptitle(title,fontsize=13,y=.995)
    fig.tight_layout(rect=[0,0.03,1,.93])
    fig.text(.025,.010,'Source / method: '+source[:165],fontsize=7,alpha=.7)
    target=OUT/'figures'/name
    fig.savefig(str(target)+'.png',dpi=300,bbox_inches='tight',facecolor='white')
    fig.savefig(str(target)+'.pdf',bbox_inches='tight',facecolor='white')
    s=(f'{title}\n\nSOURCE: {source}\n\nMETHOD: {method}\n\n'
       f'INTERPRETATION: {meaning}\n\nLIMITATION: {limitation}\n')
    (OUT/'figures'/(name+'.txt')).write_text(s,encoding='utf-8')
    FIGURES.append(dict(figure=name,title=title,source=source,
                        method=method,meaning=meaning,limitation=limitation))
    plt.show()  # nbagg keeps the displayed figure interactive; deliberately do not plt.close(fig).

METHODS={'encoded_photodiode':'Modern BV-encoded PD',
         'legacy_frame_modulo':'Legacy frame-modulo'}
mask_event=(df['event_p95_abs_ms'].notna() &
            df['event_comparison_status'].astype(str).str.startswith(('modern_','legacy_')) &
            (df['event_coverage']>=.75))
measured=df.loc[mask_event].copy()
measured['relative_p95_frames']=measured['event_p95_abs_ms']/measured['event_frame_reference_ms']
print('Sessions with ≥75% uniquely matched events:',len(measured))
if len(df):
    if FOCUS_SESSION not in event_arrays or not len(event_arrays[FOCUS_SESSION].get('event_residual_ms',[])):
        candidates=[x for x in df['session_id'].astype(str)
                    if len(event_arrays.get(x,{}).get('event_residual_ms',[]))]
        if candidates: FOCUS_SESSION=candidates[0]
print('Detailed example:',FOCUS_SESSION)

## 2. Dataset-wide distributions for Kaspar

### Figure 1 — Coverage by method and modality

**Question:** How many sessions have modern encoded photodiode events, older frame-modulo alignment, or insufficient method metadata? This determines what comparisons are defensible. The denominator is the full selected registry cohort.

In [ ]:
fig,ax=plt.subplots(figsize=(10,4.5))
tab=pd.crosstab(df['modality'],df['alignment_pathway'])
if len(tab):
    tab.plot.bar(stacked=True,ax=ax,width=.72)
    ax.legend(title='Original alignment pathway',loc='upper left',fontsize=8)
    ax.set(xlabel='Modality',ylabel='Number of sessions')
    ax.tick_params(axis='x',rotation=18)
else:_none(ax,'No registry sessions')
savefig(fig,'01_alignment_coverage','Recording coverage by modality and synchronization pathway',
        'VIPSessionRegistry sessions + analysis/qc/behavior/alignment_meta.json + original bonsai_event_log.csv/alignment_method',
        'Crosstab of pathway and modality; known reuse labels resolved from original event log.',
        'Shows which alignment methods can support modern held-out versus legacy in-fit diagnostics.',
        'File and method coverage is not itself proof of successful alignment.')

### Figure 2 — Across-session HARP PD event-timing residuals

**Question:** When BonVision says the photodiode should transition (after existing correction), how far is that prediction from the HARP-detected analog edge? Compare the p95 absolute residual distribution across sessions and acquisition methods. **Modern and legacy results are displayed separately** because one is encoded-photodiode based and the other is frame-modulo fit consistency.

In [ ]:
fig,axs=plt.subplots(1,2,figsize=(11,4.5),sharex=True)
for ax,(path,label) in zip(axs,METHODS.items()):
    g=measured.loc[measured['alignment_pathway']==path]
    x=_nums(g['event_p95_abs_ms'])
    if len(x):
        ax.hist(x,bins=min(18,max(5,int(np.sqrt(len(x))*2))),edgecolor='k')
        ax.axvline(REF_MS,ls='--',color='0.25',label=f'{DISPLAY_HZ_REFERENCE:g}-Hz reference')
        ax.legend(fontsize=8,loc='best')
        ax.text(.35,.8,f'n={len(x)} sessions\nmedian={np.median(x):.2f} ms',
                transform=ax.transAxes,va='top',ha='left',fontsize=9)
    else:_none(ax,'No qualifying sessions')
    ax.set(xlabel='p95 absolute BonVision→HARP PD residual (ms)',ylabel='Sessions',title=label)
fig.tight_layout()
fig.suptitle('Per-session HARP stimulus-transition timing precision',fontsize=18,y=1.02)
name ='02_event_residual_p95_hist'
save_figure(fig,str(OUT/'figures'/name),formats=['.png','.pdf'],dpi=300)
# savefig(fig,'02_event_residual_p95_hist','Per-session HARP stimulus-transition timing precision',
#         'bonsai_event_log.csv/Frame corrected_timestamps; HARP photodiode.pkl analog transitions; pathway metadata',
#         'For each session uniquely match predicted and detected edges; histogram p95 absolute residual, including signed bias.',
#         'Left and right panels give session distributions with a one-frame visual reference.',
#         'Both are conditional on the existing alignment model; modern held-out predictions provide stronger evidence below.')

### Figure 3 — Independent-of-fitted-event-anchor *predictive* evidence (modern sessions)

For modern encoded-photodiode sessions, the earlier audit's **held-out matched-edge interpolation** estimates residuals when every fifth paired photodiode rise is withheld as a fit anchor. Plot these separately from legacy fit residuals. Periodic matching ambiguity must be reviewed: a small holdout residual does not uniquely determine the correct whole-cycle offset.

In [ ]:
fig,axs=plt.subplots(1,2,figsize=(11,4.6))
g=df.loc[(df['alignment_pathway']=='encoded_photodiode') & df['bv_harp_holdout_p95_ms'].notna()].copy()
x=_nums(g['bv_harp_holdout_p95_ms'])
if len(x):
    axs[0].hist(x,bins=40
#                 min(18,max(5,int(np.sqrt(len(x))*2)))
                ,
                edgecolor='white')
    axs[0].axvline(REF_MS,ls='--',color='.25')
    axs[0].set(xlabel='Held-out p95 absolute residual (ms)',ylabel='Sessions')
else:_none(axs[0],'No modern held-out diagnostics')
if len(g):
    for status,sub in g.groupby('bv_harp_offset_pairing_status',dropna=False):
        axs[1].scatter(_nums(sub['bv_harp_holdout_p95_ms']),_nums(sub['bv_harp_holdout_n']),
                       label=str(status),alpha=.75)
    axs[1].axvline(REF_MS,ls='--',color='.25')
    axs[1].set(xlabel='Held-out p95 absolute residual (ms)',ylabel='Held-out edges',yscale='log')
    axs[1].legend(fontsize=7)
else:_none(axs[1],'No modern held-out diagnostics')
savefig(fig,'03_modern_holdout','BonVision→HARP prediction on withheld photodiode edges',
        'Modern BV encoded transitions; original HARP photodiode detector; held-out edge interpolation from t03_session_sync.py',
        'Withhold every fifth matched rise from interpolation; score predicted versus observed withheld rises.',
        'Shows predictive precision and number of tests per session, labeling ambiguous periodic edge matches.',
        'Matching association is still chosen upstream; low interpolation error cannot resolve a whole-period origin shift.')

### Figure 4 — Fraction of HARP transitions within one frame

This is the easiest quantitative presentation of *typical* precision. A recording with 95% of its events inside the frame reference has `fraction_within_frame≥0.95`. The fraction uses the **absolute residual**, not only jitter. The denominator is matched BonVision-to-HARP transitions; matching coverage is shown alongside it to prevent spuriously good results from a handful of events.

In [ ]:
fig,axs=plt.subplots(1,2,figsize=(11,4.6))
for label,g in measured.groupby('alignment_pathway'):
    axs[0].hist(_nums(g['event_fraction_within_frame']),bins=np.linspace(0.85,1.05,150),
                alpha=.65,label=METHODS.get(label,label))
    axs[1].scatter(_nums(g['event_coverage']),_nums(g['event_fraction_within_frame']),
                   alpha=.75,label=METHODS.get(label,label),s=25)
if len(measured):
    axs[0].legend(fontsize=8);axs[1].legend(fontsize=8)
    axs[1].axhline(.95,ls='--',color='.25')
    axs[1].axvline(.75,ls=':',color='.45')
else:
    for ax in axs:_none(ax,'No qualifying sessions')
axs[0].set(xlabel='Fraction of matched events with |residual| < one reference frame',ylabel='Sessions')
axs[1].set(xlabel='One-to-one event coverage',ylabel='Fraction within one reference frame',xlim=(0,1.02),ylim=(0,1.02))
savefig(fig,'04_within_frame_fraction','Proportion of predicted stimulus transitions detected within one frame',
        'Corrected BonVision frame transitions versus HARP analog photodiode edges; estimated or assumed display frame reference',
        'Absolute event residuals; each session supplies one fraction; only ≥75% matching-coverage sessions included.',
        'Makes the sub-frame timing consistency claim quantitative without hiding unmatched events.',
        'Frame values are estimated/assumed, not validated thresholds; fitted-model agreement is not an end-to-end T0.3 pass.')

### Figure 5 — Session-balanced empirical CDF of event error

Pooled events from long sessions can dominate histograms. This panel instead averages the **within-session empirical cumulative distribution** of `|BonVision − HARP PD| / frame_period`, contributing equal weight per recording. A sharp rise before one frame supports reproducibility across recordings; curves are separated by pathway.

In [ ]:
measured.keys()

In [ ]:
fig,ax=plt.subplots(figsize=(10,4.5))
grid=np.linspace(0,3,301)
for method,label in METHODS.items():
    curves=[]
    for _,r in measured.loc[measured['alignment_pathway']==method].iterrows():
        sid=r['session_id'];e=np.asarray(event_arrays.get(sid,{}).get('event_residual_ms',[]),float)
        frame=float(r['event_frame_reference_ms'])
        if len(e) and np.isfinite(frame) and frame>0:
            vals=np.sort(abs(e)/frame)
            curves.append(np.searchsorted(vals,grid,side='right')/len(vals))
    if curves:
        yy=np.mean(curves,axis=0)
        ax.plot(grid,yy,label=f'{label} (n={len(curves)} sessions)',lw=2)
if not ax.lines:_none(ax,'No event-level measurements')
else:
    ax.axvline(1,ls='--',color='.35',label='One frame')
    ax.legend(fontsize=8)
ax.set(xlabel='Absolute event-timing residual (display frames)',ylabel='Mean within-session cumulative fraction',ylim=(0,1.02))
savefig(fig,'05_session_balanced_ecdf','Event error in units of display frames: session-balanced CDF',
        'Per-event matched corrected BonVision and HARP photodiode times; per-session event_frame_reference_ms',
        'Compute per-session empirical CDF then average across sessions within each pathway.',
        'At x=1, the y value is the mean fraction of matched events within one reference frame.',
        'Each session has equal weight; matching selection and fitted correction remain non-independent.')

### Figure 6 — Signed event-timing bias and 95th-percentile error across sessions

The fitted `intercept` is not a physiological lag. A more relevant post-correction quantity is the **signed median of matched BonVision–HARP photodiode event residuals** (y-axis), plotted against absolute p95 residual (x-axis). A small p95 and near-zero signed median are desirable, but still depend on the alignment model.

In [ ]:
fig,ax=plt.subplots(figsize=(10,4.8))
for label,g in measured.groupby('alignment_pathway'):
    ax.scatter(_nums(g['event_p95_abs_ms']),_nums(g['event_median_ms']),
               s=28,alpha=.72,label=METHODS.get(label,label))
if len(measured):
    ax.axvline(REF_MS,ls='--',color='.3')
    ax.axhline(0,color='.5',lw=.8)
    ax.legend(fontsize=8)
else:_none(ax,'No qualifying sessions')
ax.set(xlabel='p95 absolute BV−HARP PD residual (ms)',ylabel='Signed median BV−HARP PD residual (ms)')
savefig(fig,'06_event_bias_and_jitter','Event detection precision versus systematic signed bias',
        'BonVision corrected frame timestamps minus HARP photodiode analog rise/edge detections, grouped by method',
        'Per session: signed median residual versus p95 absolute residual.',
        'Distinguishes narrow-but-biased timing from both low-bias and low-dispersion alignment.',
        'The corrected BonVision clock was calibrated using photodiode edges; these are conditional fit residuals.')

### Figure 7 — Residual change over a session and among sessions

**Question:** Does stimulus-event agreement deteriorate through a 30-min recording? Compute last-quartile minus first-quartile median corrected-BV-to-HARP event residual. This is *measured residual drift*, not the potential 18-ms duplicate-sample effect in legacy voltage extraction.

In [ ]:
fig,axs=plt.subplots(1,2,figsize=(11,4.4))
for label,g in measured.groupby('alignment_pathway'):
    vals=_nums(g['event_first_vs_last_median_ms'])
    axs[0].hist(vals,bins=min(15,max(5,int(np.sqrt(len(vals))*2))),alpha=.65,label=METHODS.get(label,label))
    axs[1].scatter(_nums(g['event_match_n']),vals,s=25,alpha=.7,label=METHODS.get(label,label))
if len(measured):
    axs[0].legend(fontsize=8);axs[1].legend(fontsize=8)
    axs[0].axvline(0,color='.4');axs[1].axhline(0,color='.4')
else:
    for ax in axs:_none(ax,'No qualifying sessions')
axs[0].set(xlabel='Late − early median BV−HARP PD residual (ms)',ylabel='Sessions')
axs[1].set(xlabel='Matched event count',ylabel='Late − early residual median (ms)',xscale='log')
savefig(fig,'07_event_residual_drift','Session-scale stability of stimulus-event timing',
        'Matched corrected BonVision transitions and HARP photodiode edges across each recording',
        'Median residual in final versus first quarter of each session; display distribution and dependence on event count.',
        'Quantifies change in behavioral-event timing accuracy through the experiment.',
        'Does not measure SLAP2 source-sample clock drift; discrete phase ambiguities can still exist.')

### Figure 8 — Stimulus onset relative to the HARP acquisition cycle

This measures the **actual photodiode event-to-nearest DI3 rising-edge phase**. A 20.67-ms cycle permits a nearest-edge phase up to approximately ±10.34 ms *by construction*. These numbers demonstrate where stimuli fall in the digital acquisition clock, **not** an error in their timestamp or a direct source sample timestamp. Source data must share the native HARP clock index; the old audit labels this comparison conditional until the shared origin is validated.

In [ ]:
fig,axs=plt.subplots(1,2,figsize=(11,4.6))
g=df.loc[df['harp_photodiode_to_di3_p95_abs_phase_ms'].notna()].copy()
if len(g):
    for label,x in g.groupby('modality'):
        axs[0].hist(_nums(x['harp_photodiode_to_di3_median_phase_ms']),
                    bins=15,alpha=.45,label=label)
        axs[1].scatter(_nums(x['di3_period_ms']),_nums(x['harp_photodiode_to_di3_p95_abs_phase_ms']),
                       s=25,alpha=.7,label=label)
    axs[0].legend(fontsize=7);axs[1].legend(fontsize=7)
else:
    for ax in axs:_none(ax,'No conditional shared-clock event phase available')
axs[0].set(xlabel='Median signed PD−nearest DI3 phase (ms)',ylabel='Sessions')
axs[1].set(xlabel='DI3 cycle period (ms)',ylabel='p95 absolute PD−DI3 phase (ms)')
savefig(fig,'08_photodiode_di3_phase','Physical HARP photodiode transitions in the DI3 acquisition clock',
        'Original HARP photodiode.pkl index and HARP_df.csv DI3 rising-edge times in the same declared time origin',
        'For each measured HARP PD rise within the DI3 recording, subtract its nearest HARP DI3 rise.',
        'Demonstrates hardware clock phase coverage at stimulus events, separated by imaging modality.',
        'Nearest-edge phase is bounded by half the DI3 period and is NOT an end-to-end physiology timing error; index origin remains conditional.')

### Figure 9 — Measurement resolution and DI3 clock jitter

HARP's analog photodiode sampling interval provides a **time-discretization scale**, not ground-truth detector error. DI3 adjacent rising-edge intervals inside acquired epochs quantify clock cadence. Imaging-off gaps are excluded from the interval jitter statistic; unequal SLAP2 and Bonsai recording durations are *not* treated as dropped pulses.

In [ ]:
fig,axs=plt.subplots(1,2,figsize=(11,4.3))
for label,g in df.groupby('alignment_pathway'):
    samp=_nums(g['harp_pd_sample_dt_median_ms'])
    samp=samp[np.isfinite(samp)]
    if len(samp):axs[0].hist(samp,bins=12,alpha=.55,label=label)
jitter=_nums(df['di3_p95_abs_jitter_us']);jitter=jitter[np.isfinite(jitter)]
if len(jitter):axs[1].hist(jitter,bins=min(20,max(6,int(np.sqrt(len(jitter))*2))),edgecolor='white')
else:_none(axs[1],'No complete DI3 clock diagnostics')
if axs[0].patches:axs[0].legend(fontsize=8)
else:_none(axs[0],'No analog sampling metadata')
axs[0].set(xlabel='Analog photodiode sample interval (ms)',ylabel='Sessions')
axs[1].set(xlabel='p95 |DI3 period − median| within epochs (µs)',ylabel='Sessions')
savefig(fig,'09_hardware_time_resolution','Acquisition-event sampling resolution and DI3 interval jitter',
        'Native index of HARP photodiode.pkl; digital rising edges from HARP_df.csv/DI3',
        'Histogram of analog sampling intervals and p95 DI3 inter-rise interval deviation (acquisition-on intervals only).',
        'Characterizes the measurement resolution and regularity of the physical HARP acquisition clock.',
        'Analog sample interval alone is not PD detection bias; DI3 interval jitter does not prove source-line mapping.')

### Figure 10 — Recording-by-recording measurement availability and T0.3 disposition

A scientifically useful audit includes missing data. There are two separate outputs: **behavioral event-fit consistency** relative to the display-frame reference, and **independently validated SLAP2-source-to-stimulus acceptance**. Neither missing metadata nor a good in-fit photodiode residual can silently become an independent PASS.

In [ ]:
fig,axs=plt.subplots(1,2,figsize=(11,4.5))
for ax,col,heading in [(axs[0],'behavior_event_gate','Behavioral event consistency'),
                       (axs[1],'t03_status','End-to-end independent T0.3')]:
    vc=df[col].fillna('NOT_MEASURED').value_counts()
    if len(vc):
        ax.barh(vc.index.astype(str),vc.values)
        for i,v in enumerate(vc):ax.text(v+.2,i,str(v),va='center',fontsize=9)
        ax.set(xlabel='Sessions',title=heading)
    else:_none(ax,'No data')
savefig(fig,'10_acceptance_and_coverage','Sub-frame fit consistency is not equivalent to independent T0.3 acceptance',
        'Full registry, corrected BonVision↔HARP photodiode event results, independent source-clock evidence CSV if supplied',
        'Left: event-fit residual below/above estimated one-frame reference. Right: independent imaging↔stimulus acceptance.',
        'Makes clear exactly which claim the currently available metadata can support.',
        'Independent PASS requires verified SLAP2 source line/DI3 clock association and verified frame period.')

### Figure 16 — Across-session timing precision versus acquisition date

A longitudinal **line/scatter graph** avoids hiding whether timing performance improved or deteriorated over acquisition campaigns. Session IDs provide session dates (`YYYY-MM-DD`) in the usual registry naming convention; sessions without a parsable date are omitted from the x-axis (they remain in the denominator elsewhere). p95 absolute residuals are shown in units of each session's estimated/reference display frame period, and separated by method. The line connecting sessions is a visual ordering, **not** interpolation or smoothing.

In [ ]:
fig,ax=plt.subplots(figsize=(11,4.4))
s=measured.copy()
s['session_date_from_id']=pd.to_datetime(s['session_id'].astype(str).str.extract(r'(\d{4}-\d{2}-\d{2})')[0],errors='coerce')
for method,label in METHODS.items():
    g=s.loc[(s['alignment_pathway']==method)&s['session_date_from_id'].notna()].sort_values('session_date_from_id')
    if len(g):ax.plot(g['session_date_from_id'],g['relative_p95_frames'],'o-',ms=4,lw=.85,alpha=.65,label=label)
if ax.lines:
    ax.axhline(1,ls='--',color='.25',label='One display frame reference')
    ax.legend(fontsize=8)
else:_none(ax,'No dated sessions with per-event QC results')
ax.set(xlabel='Acquisition date (from registry session ID)',ylabel='Per-session p95 |BV−HARP PD| (display frames)')
savefig(fig,'16_event_error_across_dates','Stimulus timing consistency across recording campaigns',
        'Registry session IDs; corrected BonVision event times; measured HARP analog photodiode edges; session frame reference',
        'Connect per-session p95 absolute residuals in chronological acquisition order; lines are only visual guides.',
        'Checks for systematic changes in alignment quality between sessions or acquisition campaigns.',
        'No interpolation or smoothing. The vertical metric is behavioral fit consistency, not independently validated SLAP2 timing.')

### Figure 17 — BonVision/HARP fitted clock-origin offsets (not latencies)

The alignment model's fitted intercept is an **offset between clock coordinate origins**. It can vary by seconds while the residual alignment error remains sub-frame, so it must not be interpreted as physiology/stimulus lag. We show the distribution chiefly to document gross changes or suspicious multimodality in clock origins.

In [ ]:
fig,axs=plt.subplots(1,2,figsize=(11,4.3))
for method,label in METHODS.items():
    g=df.loc[(df['alignment_pathway']==method)&df['bv_harp_intercept_ms'].notna()]
    v=_nums(g['bv_harp_intercept_ms'])
    if len(v):
        axs[0].hist(v/1000,bins=min(18,max(5,int(np.sqrt(len(v))*2))),alpha=.55,label=label)
        axs[1].scatter(v/1000,_nums(g['bv_harp_slope']),alpha=.65,s=25,label=label)
if axs[0].patches:
    axs[0].legend(fontsize=8);axs[1].legend(fontsize=8)
else:
    for ax in axs:_none(ax,'No original clock-fit parameters available')
axs[0].set(xlabel='Fitted BV→HARP origin intercept (s)',ylabel='Sessions')
axs[1].set(xlabel='Fitted origin intercept (s)',ylabel='Fitted BV→HARP slope')
savefig(fig,'17_bv_harp_clock_origin','BonVision-to-HARP model coordinate offsets, shown separately from timing errors',
        'analysis/qc/behavior/alignment_meta.json/intercept and slope in repository BV-to-HARP mapping',
        'Plot saved affine clock-origin intercepts and slopes, preserving modern versus legacy pathway labels.',
        'Documents offset and clock scale variation without confusing intercept with physiological latency.',
        'Clock-origin parameters are coordinate conventions; piecewise models have additional warp parameters; absent reused metadata are missing.')

## 3. Representative recording: event-level distributions and elapsed-time plots

The next figures are designed as slide panels or supplementary figures for one named recording. The selected session must have the requisite assets. Nothing is smoothed; points show individual events. 5-min bin medians (when plotted) are simply explicit temporal summaries.

In [ ]:
focus=df.loc[df['session_id'].astype(str)==FOCUS_SESSION]
focus_row=focus.iloc[0] if len(focus) else None
focus_events=event_arrays.get(FOCUS_SESSION,{})
focus_hardware=event_detail.get(FOCUS_SESSION,{})
print('Focus:',FOCUS_SESSION)
if focus_row is not None:
    display(focus[[c for c in ['session_id','modality','alignment_method',
                   'event_comparison_status','event_phase_source',
                   'event_match_n','event_coverage','event_p95_abs_ms',
                   'harp_pd_di3_phase_status','t03_status'] if c in focus]])

### Figure 11 — Event residual histogram and 5th–95th percentile range

Signed event residuals directly answer whether detected HARP photodiode events agree with their corrected BonVision predictions. The dashed limits show ±one reference display frame. Compare the *fraction within frame* and *matching coverage*, not merely the peak of the distribution.

In [ ]:
fig,ax=plt.subplots(figsize=(10,4.4))
er=np.asarray(focus_events.get('event_residual_ms',[]),float)
ref_ms=float(focus_row['event_frame_reference_ms']) if focus_row is not None else REF_MS
if len(er):
    ax.hist(er,bins=45,edgecolor='white',alpha=.9)
    for sign in [-1,1]:ax.axvline(sign*ref_ms,color='0.25',ls='--')
    ax.axvline(np.median(er),color='0.35',lw=1.5,label='Median residual')
    ax.legend(fontsize=8)
    print(f'Focus: n={len(er)}, |err|p95={np.percentile(abs(er),95):.3f} ms, '
          f'within-frame={np.mean(abs(er)<ref_ms):.1%}, frame ref={ref_ms:.3f} ms')
else:_none(ax,'No corrected BonVision↔HARP PD event comparison for this session')
ax.set(xlabel='Corrected BonVision event − HARP PD edge (ms)',ylabel='Matched stimulus events')
savefig(fig,'11_focus_event_error_hist','Example recording: physical stimulus-event timing residuals',
        f'{FOCUS_SESSION}: corrected Bonsai event-log timestamps; HARP analog PD edges',
        'One-to-one event matches; histogram of signed prediction-minus-detection residual.',
        'Shows the signed lag and distribution of individual stimulus transitions for a real session.',
        'Modern encoded PD samples were used to calibrate the same clock; legacy frames are model-fit checks.')

### Figure 12 — Event residuals over recording time

This reveals shifts or systematic evolution of event detection alignment during the recording that could be hidden in a global histogram. One dot per observed event. No smoothing/filtering. The dashed lines are the nominal/estimated display-frame reference.

In [ ]:
fig,ax=plt.subplots(figsize=(11,4.4))
t=np.asarray(focus_events.get('event_bv_time_s',[]),float)
if len(t)==len(er) and len(er):
    ax.scatter(t/60,er,s=8,alpha=.5)
    ax.axhline(ref_ms,ls='--',color='.3');ax.axhline(-ref_ms,ls='--',color='.3')
    ax.axhline(0,color='.5',lw=.7)
    ax.set(xlabel='Corrected BonVision time (min)',ylabel='BV−HARP PD event residual (ms)')
else:_none(ax,'No matched event time series')
savefig(fig,'12_focus_event_residual_timeseries','Example recording: timing residuals throughout acquisition',
        f'{FOCUS_SESSION}: per-event corrected BonVision timestamps and detected HARP photodiode edges',
        'Plot the signed residual of every matched event against corrected BonVision elapsed time.',
        'Tests whether event timing remains accurate and stable through the recording.',
        'Timing residuals are not a line-index-to-HARP drift measurement; dashed frame is a reference only.')

### Figure 13 — HARP stimulus phase versus SLAP2 DI3 clock over recording

Actual photodiode transitions and DI3 rises both come from HARP acquisition files. The y-axis is **signed event-to-nearest-pulse phase**, which can wrap when the display rhythm and acquisition clock are incommensurate. It is not physiological latency or clock-registration error.

In [ ]:
fig,ax=plt.subplots(figsize=(11,4.2))
phase=focus_hardware.get('harp_pd_di3',{})
pp=np.asarray(phase.get('phase_ms',[]),float)
pt=np.asarray(phase.get('time_s',[]),float)
if len(pp)==len(pt) and len(pp):
    ax.scatter((pt-pt[0])/60,pp,s=8,alpha=.55)
    ax.axhline(0,color='.5',lw=.8)
    ax.set(xlabel='Time from first included HARP PD event (min)',ylabel='PD−nearest DI3 rise (ms)')
else:_none(ax,'Native HARP PD↔DI3 phase not available / origin not established')
savefig(fig,'13_focus_physical_event_di3_phase','Example recording: HARP stimulus events in SLAP2 acquisition-cycle time',
        f'{FOCUS_SESSION}: HARP analog photodiode.pkl and HARP_df.csv/DI3, unmodified acquisition indices',
        'Subtract nearest DI3 rising-edge time from each HARP photodiode rising-edge time within recorded epochs.',
        'Shows the phase at which each physical display transition occurred relative to the acquisition cycles.',
        'Nearest-edge phase is not a synchronization error; same-clock index origin and per-DMD line mapping remain to be verified.')

### Figure 14 — Raw digital clock and physical stimulus event raster (short interval)

This is an intuitive presentation: HARP DI3 cycle-clock rising edges are shown alongside HARP photodiode transition markers in a short time window. It visualizes the hardware relationship without pretending that individual fluorescence sample times have already been reconstructed.

In [ ]:
fig,ax=plt.subplots(figsize=(11,3.7))
pulses=np.asarray(focus_hardware.get('di3_rising_edges',[]),float)
pd_h=np.asarray(focus_hardware.get('harp_pd_rising_edges_absolute',[]),float)
if len(pulses) and len(pd_h):
    k=min(5,len(pd_h)-1);start=float(pd_h[k])-.05;end=start+.35
    dp=pulses[(pulses>=start)&(pulses<=end)]
    hp=pd_h[(pd_h>=start)&(pd_h<=end)]
    if len(dp):ax.vlines(1000*(dp-start),.0,.8,label='HARP DI3 (SLAP2 cycle rises)')
    if len(hp):ax.vlines(1000*(hp-start),1.0,1.8,color='tab:orange',lw=2,label='HARP analog photodiode rises')
    ax.set(yticks=[.4,1.4],yticklabels=['DI3 cycle','Actual PD event'],
           xlabel='Time from chosen HARP window start (ms)',ylim=(-.1,2))
    ax.legend(fontsize=8)
else:_none(ax,'No common-window HARP DI3 + photodiode event timestamps')
savefig(fig,'14_focus_raw_hardware_events','Actual stimulus event versus SLAP2 acquisition pulses in HARP time',
        f'{FOCUS_SESSION}: native HARP_df.csv/DI3 and photodiode.pkl index',
        'Vertical tick marks locate unaltered HARP digital rising edges and analog photodiode rises over a 350-ms window.',
        'Directly visualizes the acquisition-clock and stimulus-event relationship in HARP time.',
        'Cannot show individual DMD source-sample times without a verified line-index-to-DI3 mapping.')

### Figure 18 — Direct BonVision → HARP → DI3 event-chain comparison across sessions

For each **matched event**, the analysis can place both corrected BonVision's predicted transition and HARP's actually detected photodiode transition into the **DI3 cycle time coordinate** via the native HARP timestamp. The resulting nearest-DI3 phases are plotted here in units of ms; session labels are separated by modern versus legacy alignment. Because the phase is modulo the cycle period, its p95 **does not equal an imaging-stimulus timing error**. The same-cycle agreement fraction is more diagnostic of the residual mapping than just the phase magnitude. Origin equivalence currently remains a conditional assumption from the source timestamps and is flagged in the exported table.

In [ ]:
fig,axs=plt.subplots(1,2,figsize=(11,4.4))
for method,label in METHODS.items():
    g=df.loc[(df['alignment_pathway']==method)&df['event_bv_to_di3_p95_abs_phase_ms'].notna()]
    if len(g):
        axs[0].hist(_nums(g['event_bv_to_di3_p95_abs_phase_ms']),bins=14,alpha=.5,label=label)
        axs[1].scatter(_nums(g['event_bv_to_di3_p95_abs_phase_ms']),
                       _nums(g['event_same_nearest_di3_cycle_fraction']),
                       alpha=.7,label=label,s=25)
if axs[0].patches:axs[0].legend(fontsize=8);axs[1].legend(fontsize=8)
else:
    for ax in axs:_none(ax,'No conditional BonVision → HARP → DI3 timing chain')
axs[0].set(xlabel='p95 |corrected BonVision event−nearest DI3 rise| (ms)',ylabel='Sessions')
axs[1].set(xlabel='p95 absolute predicted-event phase (ms)',ylabel='Fraction of paired BV and PD events nearest same DI3 cycle',ylim=(0,1.02))
savefig(fig,'18_bv_harp_di3_chain','Matched BonVision events and detected stimuli relative to HARP DI3 clock',
        'Corrected bonsai_event_log.csv event times; HARP photodiode.pkl analog edges; original HARP_df.csv/DI3 rising edges',
        'Use the photodiode index origin to put corrected BV events and observed HARP PD events on the DI3 clock; nearest-pulse phase and same-cycle fraction.',
        'A direct event-level linkage across all three measured data streams, indicating where predicted and actual stimulus onsets occur in acquisition cycles.',
        'Source-HARP common origin is conditional; closest DI3 cycle is NOT proven source frame/line time, and the p95 phase is not synchronization error.')

### Figure 19 — Focus session: corrected BonVision and physical HARP event relative to DI3

Each observed stimulus event has two possible coordinates in HARP time: the **predicted** BonVision event (after correction) and the **measured** photodiode transition. Show their pulse phases side by side for the first 100 matched events. Their separation can reflect residual alignment, and wraps reflect the nearest-pulse coordinate's modulo-cycle nature. This is the most direct visual for the three-clock measurement chain that does not invent a per-ROI sample origin.

In [ ]:
fig,ax=plt.subplots(figsize=(11,4.3))
pp=np.asarray(focus_events.get('event_nearest_di3_phase_ms',[]),float)
bp=np.asarray(focus_events.get('event_bv_to_di3_phase_ms',[]),float)
nn=min(len(pp),len(bp),100)
if nn:
    indices=np.arange(nn)
    ax.scatter(indices,bp[:nn],label='BonVision corrected event → DI3 phase',s=18)
    ax.scatter(indices,pp[:nn],label='HARP observed photodiode → DI3 phase',s=18,alpha=.7,marker='x')
    ax.axhline(0,color='.5',lw=.7)
    ax.legend(fontsize=8)
    ax.set(xlabel='Matched stimulus transition (first 100 within DI3 window)',
           ylabel='Signed phase to nearest DI3 rise (ms)')
else:_none(ax,'Common-origin event-to-DI3 phase was unavailable')
savefig(fig,'19_focus_bv_pd_di3_phases','Example recording: measured stimulus and corrected BonVision in DI3 cycle time',
        f'{FOCUS_SESSION}: corrected BonVision event times + physical HARP PD edges + DI3 cycle rises',
        'For matched stimuli, compare nearest DI3 phase for each corrected BonVision prediction and each actual HARP photodiode event.',
        'Displays whether event corrections place BonVision predictions at the same acquisition-cycle phase as the measured physical transitions.',
        'Phase wraps are natural and not error. No inferred GIAnT frame_line_idx or legacy voltage sample time is shown without a validated physical anchor.')

## 4. Supporting extraction-integrity checks and independent acceptance

The source-boundary observation (a possible cumulative ~18-ms duplicate-sample coordinate discrepancy in one voltage session) is a **separate processing-integrity QC finding**, not direct stimulus-alignment drift. GIAnT `frame_line_idxs` represent acquisition lines, whereas legacy downstream `timebase_sec` is a reconstructed timebase. The optional source scan is deliberately separate from the behavioral-event pass.

In [ ]:
fig,axs=plt.subplots(1,2,figsize=(11,4.2))
src=df.loc[pd.to_numeric(df.get('source_boundaries_tested',pd.Series(np.nan,index=df.index)),errors='coerce')>0]
if len(src):
    denom=_nums(src['source_boundaries_tested'])
    numer=_nums(src['source_identical_samples'])
    axs[0].hist(numer/denom,bins=np.linspace(0,1,11),edgecolor='white')
else:_none(axs[0],'Source voltage H5 scan disabled or no source trials')
axs[0].set(xlabel='Identical endpoint samples / tested source trial boundaries',ylabel='Voltage sessions')
gi=pd.to_numeric(df.get('giant_nonincreasing_frame_line_steps',pd.Series(np.nan,index=df.index)),errors='coerce')
if gi.notna().any():
    axs[1].hist(gi.dropna(),bins=12,edgecolor='white')
else:_none(axs[1],'GIAnT frame_line_idxs scan disabled or unavailable')
axs[1].set(xlabel='Nonincreasing GIAnT frame_line_idx steps (count)',ylabel='GIAnT sessions')
savefig(fig,'15_extraction_coordinate_qc','Supporting QC: source extraction line-coordinate and endpoint behavior',
        'Voltage dendriticVoltageTraces HDF5 /traces/trial attrs and raw endpoint samples; GIAnT ExperimentSummary HDF5 Path*/frame_info/frame_line_idxs',
        'Separate distributions of equal repeated voltage endpoints and nonincreasing GIAnT physical line-index steps.',
        'Documents prevalence of source-processing timing conventions across sessions and modalities.',
        'Neither finding alone determines absolute HARP↔SLAP2 timing error. Turn on SCAN_EXTRACTION to populate.')

## 5. Dataset tables, provenance manifest, and Kaspar slide notes

**Suggested presentation order:** (1) coverage, (2) modern held-out distribution, (3) behavioral event fit histogram and within-frame fraction, (4) hardware PD–DI3 phase, (5) representative residual-vs-time, (6) independent acceptance and open source-anchor requirement. Maintain separate slides or legends for legacy frame-modulo sessions.

**Validation file schema:** `session_id,imaging_to_stimulus_lag_ms,p95_abs_error_ms,verified_frame_period_ms,method,evidence_path,independent_validation`. Its evidence must explicitly establish SLAP2 source lines/epochs/DMD clock against DI3 and HARP PD events, including systematic offset and jitter. `independent_validation=true` is a report approval flag, not a way to manufacture evidence.

In [ ]:
columns=['session_id','subject_id','modality','alignment_pathway','alignment_method',
         'event_comparison_status','event_match_n','event_candidate_n','event_coverage',
         'event_median_ms','event_p95_abs_ms','event_p99_abs_ms',
         'event_fraction_within_frame','event_first_vs_last_median_ms',
         'event_frame_reference_ms','event_frame_reference_type',
         'bv_harp_holdout_p95_ms','bv_harp_holdout_n','bv_harp_offset_pairing_status',
         'harp_pd_sample_dt_median_ms','harp_photodiode_to_di3_median_phase_ms',
         'harp_photodiode_to_di3_p95_abs_phase_ms','harp_pd_di3_phase_status',
         'di3_period_ms','di3_p95_abs_jitter_us','di3_n_epoch_gaps',
         'source_boundaries_tested','source_identical_samples',
         'giant_nonincreasing_frame_line_steps','imaging_to_stimulus_lag_ms',
         'imaging_to_stimulus_p95_abs_error_ms',
         'behavior_event_gate','t03_status','t03_reason','analysis_errors','evidence_sources']
summary=df[[c for c in columns if c in df]].copy()
summary.to_csv(OUT/'T03_Kaspar_session_metrics.csv',index=False)

# Save raw event-level *timing diagnostics*, but no fluorescence traces.
event_rows=[]
for sid,details in event_arrays.items():
    err=np.asarray(details.get('event_residual_ms',[]),float)
    tt=np.asarray(details.get('event_bv_time_s',[]),float)
    observed=np.asarray(details.get('event_harp_time_s',[]),float)
    if len(tt)==len(err)==len(observed):
        event_rows.extend((sid,float(a),float(b),float(c)) for a,b,c in zip(tt,observed,err))
pd.DataFrame(event_rows,columns=['session_id','corrected_bv_event_s',
   'harp_detected_pd_event_s','bv_minus_harp_pd_ms']).to_csv(OUT/'T03_matched_event_residuals.csv',index=False)

figure_meta=pd.DataFrame(FIGURES)
figure_meta.to_csv(OUT/'T03_figure_provenance.csv',index=False)
status=df['t03_status'].value_counts(dropna=False).to_dict()
modern_sub=df.loc[df['alignment_pathway']=='encoded_photodiode','bv_harp_holdout_p95_ms'].dropna()
caption=f"""# T0.3 — Results from the local session registry\n\n- Sessions audited: **{len(df)}**.
- Session-level behavioral event residuals with ≥75% matching coverage: **{len(measured)}**.
- Modern encoded-photodiode sessions with held-out BV→HARP predictions: **{len(modern_sub)}**.
- Independent end-to-end T0.3 decisions: **{status}**.
- Absolute source line-index→HARP DI3 relationship and DMD/epoch mapping remains required for sessions without independent validation.

**Definitions:** signed per-event residual = corrected BonVision frame transition − HARP measured analog PD edge. This measures consistency after the saved alignment was already applied. HARP PD−DI3 phase measures physical stimulus onset relative to the acquisition cycle, *not* timestamp error. The one-frame visual reference is estimated or assumed and never substitutes for a verified display frame period.

**Slide figures:** `figures/*.png` and `figures/*.svg`; full per-figure provenance and interpretation in `figures/*.txt`. Missing measurements must be shown separately from scientific failures. No pipeline files were modified.\n"""
(OUT/'T03_Kaspar_briefing.md').write_text(caption,encoding='utf-8')
display(summary.head(25))
display(Markdown(caption))
print('Figures exported:',len(FIGURES),'| event rows:',len(event_rows),'| location:',OUT)

## 6. Interpretation / limitations to state explicitly

- **HARP PD detection is measured**, but its threshold-crossing precision is bounded by analog sampling, detection settings, SNR and possible display/electronics latency. A sub-frame corrected-event residual cannot by itself establish sub-frame absolute *stimulus onset* calibration.
- **Modern photodiode-encoded sessions:** the per-event residual reuses data involved in the original clock alignment, and the stored model may be piecewise; use the held-out distribution for stronger precision evidence. Whole-cycle correspondence may still be ambiguous for periodic signals.
- **Legacy frame-modulo sessions:** lacking BV-coded events, the transition class/phase can be reconstructed from corrected event timestamps. That reconstruction is a *fitted* consistency check, not an independent validation, and must be reported separately.
- **SLAP2:** DI3 is a hardware cycle clock. Mapping source-extraction line indices into its time coordinate requires verified epoch, DMD, origin, sample index and line-period information. Merely finding the nearest DI3 edge gives a bounded *phase*, **not** a sample-to-stimulus error. Modern GIAnT sample rate, voltage extraction rate and DMD cycle rate must never be treated as interchangeable.
- **No pulse losses inferred from unequal total durations:** HARP and SLAP2 start/end windows may differ by approximately 25 s; imaging-off intervals are distinguished from within-epoch unexpected gaps.
- **No automatic exclusion based on unverified metrics:** T0.3 PASS/FAIL requires independent full-chain evidence and a verified frame period, while missing evidence remains `NOT_INDEPENDENTLY_VERIFIED`.
- **No physiological trace filtering:** only the repository's existing HARP *photodiode* edge detector uses a median filter. All displayed event times are discrete timestamps; no voltage smoothing, resampling or source extraction is performed.